# Exploratory Data Analysis

Exploration of the sales panel (one store, 1,427 items, daily).
Shared logic lives in `eda_core.py`, plots in `eda_plots.py`.
The datasets used by the models are produced by `build_datasets.py`, not by this notebook.

In [ ]:
import pandas as pd

import eda_core as core
import eda_plots as plots

pd.set_option("display.max_columns", 50)

raw = core.load_panel(core.RAW_PATH)              # as delivered
df = core.load_panel(core.FULFILLED_PATH)         # complete daily grid (missing days = 0)
print(f"raw: {raw.shape}, fulfilled: {df.shape}")
df.head()

## 1. Coverage and imputation
Gaps of each (item, store) against the global date range, before the calendar grid is filled.

In [ ]:
coverage = core.coverage_report(raw)
print(f"Global range: {raw[core.DATE_COL].min().date()} -> {raw[core.DATE_COL].max().date()}")
print(f"Pairs: {len(coverage)}, incomplete: {(coverage['missing_days'] > 0).sum()}, "
      f"started late: {(coverage['days_missing_before'] > 0).sum()}, "
      f"ended early: {(coverage['days_missing_after'] > 0).sum()}")
coverage.sort_values("missing_days", ascending=False).head(10)

In [ ]:
# ADI/CV of the late starters, computed on their observed (raw) history only
late = coverage[coverage["days_missing_before"] > 0]
late_labels = core.classify_adi_cv(core.rows_for_pairs(raw, late))
late = late.merge(late_labels, on=core.KEYS)
print(late[core.LABEL_COL].value_counts())
late.sort_values("days_missing_before", ascending=False).head(10)

In [ ]:
# Zero-sales ratio per product after filling
core.sales_summary(df).sort_values("zero_rate", ascending=False).head(10)

## 2. Store closures and holidays

In [ ]:
STORE_ID = 6269
closures = core.store_closures(raw, STORE_ID)
print(f"Store {STORE_ID}: {closures['open_days']} open / {closures['total_days']} days, "
      f"{closures['closed_days']} closed ({closures['closure_rate']:.1%}), "
      f"{len(closures['periods'])} closure periods")
closures["periods"].head()

In [ ]:
core.holiday_store_sales(raw)

## 3. Catalogue

In [ ]:
print(f"Items: {df[core.ITEM_COL].nunique()}, stores: {df[core.STORE_COL].nunique()}")
print("Promotion columns:", core.promo_columns(df))
plots.plot_sku_share(core.skus_per(df, "dep_label"), kind="pie", title="Unique SKUs per department")
plots.plot_sku_share(core.skus_per(df, "cat_label"), kind="bar", title="Unique SKUs per category")

## 4. ADI/CV classification
Syntetos et al.: ADI 1.32, CV 0.49, computed on the filled series.

In [ ]:
labels = core.classify_adi_cv(df)
plots.plot_label_distribution(labels)

In [ ]:
for label in core.LABEL_ORDER:
    sub = labels[labels[core.LABEL_COL] == label]
    print(f"=== {label.upper()} ({len(sub)} products) ===")
    display(sub.sort_values("adi").head(5))

In [ ]:
plots.plot_random_sample_per_label(df, labels, seed=None)   # seed=int for a fixed sample

## 5. Sales ranking and top products
The model experiments use smooth + erratic products with total sales >= 12,500 (`top_12500.feather`).

In [ ]:
summary = core.sales_summary(df, labels)
smooth_erratic = core.select_top_products(summary, labels=("smooth", "erratic"))
plots.plot_ranked_sales(smooth_erratic, "total_sales", title="Total sales ranked by product (smooth + erratic)")
plots.plot_ranked_sales(smooth_erratic, "mean_sales", title="Mean sales ranked by product (smooth + erratic)");

In [ ]:
MIN_TOTAL_SALES, MIN_MEAN_SALES = 12500, 20
top_total = core.select_top_products(summary, min_total_sales=MIN_TOTAL_SALES)
top_mean = core.select_top_products(summary, min_mean_sales=MIN_MEAN_SALES, sort_by="mean_sales")
print(f"total sales >= {MIN_TOTAL_SALES}: {len(top_total)} | mean sales >= {MIN_MEAN_SALES}: {len(top_mean)}")
top_total.head(40).style.background_gradient(subset=["total_sales"], cmap="Blues")

In [ ]:
# Top products regardless of label
summary.sort_values("total_sales", ascending=False).head(50)

## 6. Product inspector

In [ ]:
ITEM_ID, STORE_ID = 907969, 6269
product = core.product_rows(df, ITEM_ID, STORE_ID)
plots.plot_product(df, ITEM_ID, STORE_ID)
plots.plot_product_splits_html(df, ITEM_ID, STORE_ID, show=True);

In [ ]:
active, by_flag, by_type = core.promo_summary(product)
print("Active promotion types:", active)
if active:
    display(by_flag)
    display(by_type)
    plots.plot_promo_timeline(product)

In [ ]:
# Export interactive train/val/test HTML for every top product
# for item_id, store_id in top_total[core.KEYS].itertuples(index=False):
#     plots.plot_product_splits_html(df, item_id, store_id, output_dir=f"TSPLots/{item_id}")